First, I just read in our original data:

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

giftdf = pd.read_csv('ForeignGifts_edu.csv',low_memory=False)
giftdf.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28221 entries, 0 to 28220
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   ID                          28221 non-null  int64 
 1   OPEID                       28221 non-null  int64 
 2   Institution Name            28221 non-null  object
 3   City                        28221 non-null  object
 4   State                       28221 non-null  object
 5   Foreign Gift Received Date  28221 non-null  int64 
 6   Foreign Gift Amount         28221 non-null  int64 
 7   Gift Type                   28221 non-null  object
 8   Country of Giftor           28221 non-null  object
 9   Giftor Name                 24470 non-null  object
dtypes: int64(4), object(6)
memory usage: 2.2+ MB


In [2]:
giftdf.isna().sum()

ID                               0
OPEID                            0
Institution Name                 0
City                             0
State                            0
Foreign Gift Received Date       0
Foreign Gift Amount              0
Gift Type                        0
Country of Giftor                0
Giftor Name                   3751
dtype: int64

In [3]:
giftdf.head()

,ID,OPEID,Institution Name,City,State,Foreign Gift Received Date,Foreign Gift Amount,Gift Type,Country of Giftor,Giftor Name
0,1,102000,Jacksonville State University,Jacksonville,AL,43738,250000,Monetary Gift,CHINA,NaN
1,2,104700,Troy University,Troy,AL,43592,463657,Contract,CHINA,Confucius Institute Headquarters
2,3,105100,University of Alabama,Tuscaloosa,AL,43466,3649107,Contract,ENGLAND,Springer Nature Customer Service Ce
3,4,105100,University of Alabama,Tuscaloosa,AL,43472,1000,Contract,SAUDI ARABIA,Saudi Arabia Education Mission
4,5,105100,University of Alabama,Tuscaloosa,AL,43479,49476,Contract,SAUDI ARABIA,Saudi Arabia Education Mission


As a bit of housekeeping, we will modify the date times provided in the data frame. I cannot find exact documentation on what this date was encoded as, but I am assuming this is using the Excel serial numbers; dates are stored as a certain number of days after December 30, 1899. 

In [4]:
giftdf['Foreign Gift Received Date'] = pd.to_datetime(
    giftdf['Foreign Gift Received Date'], unit='D', origin='1899-12-30'
)

In [5]:
giftdf.head()

,ID,OPEID,Institution Name,City,State,Foreign Gift Received Date,Foreign Gift Amount,Gift Type,Country of Giftor,Giftor Name
0,1,102000,Jacksonville State University,Jacksonville,AL,2019-09-30,250000,Monetary Gift,CHINA,NaN
1,2,104700,Troy University,Troy,AL,2019-05-07,463657,Contract,CHINA,Confucius Institute Headquarters
2,3,105100,University of Alabama,Tuscaloosa,AL,2019-01-01,3649107,Contract,ENGLAND,Springer Nature Customer Service Ce
3,4,105100,University of Alabama,Tuscaloosa,AL,2019-01-07,1000,Contract,SAUDI ARABIA,Saudi Arabia Education Mission
4,5,105100,University of Alabama,Tuscaloosa,AL,2019-01-14,49476,Contract,SAUDI ARABIA,Saudi Arabia Education Mission


I need to check gift types first to make aggregation easier.

In [6]:
giftdf["Gift Type"].value_counts()

Gift Type
Contract         17274
Monetary Gift    10936
Real Estate         11
Name: count, dtype: int64

In [7]:
giftdf[giftdf["Gift Type"] == "Real Estate"]

,ID,OPEID,Institution Name,City,State,Foreign Gift Received Date,Foreign Gift Amount,Gift Type,Country of Giftor,Giftor Name
9245,9246,132000,"University of California, Santa Barbara",Santa Barbara,CA,2019-01-01,39456,Real Estate,JAPAN,Mitsubishi Chemical Holdings Inc
9618,9619,135000,Colorado State University,Fort Collins,CO,2014-09-29,4312000,Real Estate,MEXICO,MIRA
23417,23418,337100,Temple University,Philadelphia,PA,2017-03-31,878139,Real Estate,JAPAN,Itochu Urban Community
23418,23419,337100,Temple University,Philadelphia,PA,2017-08-01,667555,Real Estate,JAPAN,Sublease
23419,23420,337100,Temple University,Philadelphia,PA,2017-08-01,331378,Real Estate,JAPAN,Sublease
23420,23421,337100,Temple University,Philadelphia,PA,2017-08-04,2829286,Real Estate,JAPAN,Taisei Biru Kanri K.K.
23421,23422,337100,Temple University,Philadelphia,PA,2017-08-04,1414186,Real Estate,JAPAN,Taisei Biru Kanri K.K.
23441,23442,337100,Temple University,Philadelphia,PA,2018-08-23,322314,Real Estate,JAPAN,Uninest
23452,23453,337100,Temple University,Philadelphia,PA,2019-03-31,409771,Real Estate,JAPAN,Itochu Urban Community
23457,23458,337100,Temple University,Philadelphia,PA,2019-06-30,3023538,Real Estate,ITALY,Fattura S.E.RO. CE. S.R.L.


Strange. Real estate is almost all Japan and almost all to Temple.

In [8]:
giftdf['Foreign Gift Received Date'].min()

Timestamp('2014-01-01 00:00:00')

In [9]:
giftdf['Foreign Gift Received Date'].max()

Timestamp('2020-06-30 00:00:00')

I also want to compute an average giftor rank so I am going to do this in the next few cells; for ease, I am going to just make the name of the column what I want it to be at the end; this means intermediate steps will poorly align with their names. 

In [10]:
country_gifts = giftdf.groupby("Country of Giftor")["Foreign Gift Amount"].sum().sort_values(ascending=False).reset_index()[["Country of Giftor"]]
country_gifts["average rank"] = country_gifts.index + 1

In [11]:
country_gifts.head(10)

,Country of Giftor,average rank
0,QATAR,1
1,ENGLAND,2
2,CHINA,3
3,SAUDI ARABIA,4
4,BERMUDA,5
5,CANADA,6
6,HONG KONG,7
7,JAPAN,8
8,SWITZERLAND,9
9,INDIA,10


In [12]:
giftdf = pd.merge(giftdf, country_gifts, how = "left", on = "Country of Giftor")

In [13]:
giftdf["average rank"] = giftdf["average rank"] * giftdf["Foreign Gift Amount"]

Here, I will perform our standard aggregation. Note that I will still use the original data later, but the aggregation will give us a more succint and palatable way to compare colleges.

In [14]:
grouped_gift = giftdf.groupby(["OPEID", "Institution Name"]).agg(total_donation_amount = ("Foreign Gift Amount", "sum"),
                                                                total_donations = ("Foreign Gift Amount", "size"),
                                                                total_countries = ("Country of Giftor", "nunique"),
                                                                total_contract_donations = ("Gift Type", lambda x: (x == 'Contract').sum()),
                                                                total_gift_donations = ("Gift Type", lambda x: (x == 'Monetary Gift').sum()),
                                                                total_estate_donation = ("Gift Type", lambda x: (x == 'Real Estate').sum()),
                                                                average_giftor_rank = ("average rank", "sum"),
                                                                total_institutions = ("Giftor Name", "nunique")
                                                                ).reset_index()

In [15]:
grouped_gift["average_giftor_rank"] = grouped_gift["average_giftor_rank"] / grouped_gift["total_donation_amount"] 

In [16]:
grouped_gift.head()

,OPEID,Institution Name,total_donation_amount,total_donations,total_countries,total_contract_donations,total_gift_donations,total_estate_donation,average_giftor_rank,total_institutions
0,102000,Jacksonville State University,250000,1,1,0,1,0,3.000000,0
1,104700,Troy University,463657,1,1,1,0,0,3.000000,1
2,105100,University of Alabama,23626197,63,7,58,5,0,12.498734,14
3,105200,University of Alabama at Birmingham,7190988,241,10,240,1,0,14.012374,14
4,105700,University of South Alabama,8537196,18,3,16,2,0,5.717938,15


In [17]:
uva = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]

We can quickly view some interesting metrics on the break down of UVA gifitng:

In [18]:
giftdf[giftdf["Institution Name"] == "University of Virginia"].groupby("Country of Giftor").agg(total_donation_amount = ("Foreign Gift Amount", "sum"), total_institutions= ("Giftor Name", "nunique")).sort_values(by="total_donation_amount", ascending=False)

,total_donation_amount,total_institutions
Country of Giftor,,
BANGLADESH,4228589,5
SWEDEN,4007552,5
TANZANIA,2776447,3
SWITZERLAND,2000000,2
ENGLAND,1950666,14
GUERNSEY,1500000,2
FINLAND,1008384,2
BERMUDA,900000,1
KOREA,862787,5


In [19]:
gifts_counts_countries = giftdf[giftdf["Institution Name"] == "University of Virginia"]["Country of Giftor"].value_counts().reset_index()
country_gifts[country_gifts["Country of Giftor"].isin(gifts_counts_countries["Country of Giftor"])]

,Country of Giftor,average rank
1,ENGLAND,2
4,BERMUDA,5
7,JAPAN,8
8,SWITZERLAND,9
12,FRANCE,13
13,SINGAPORE,14
17,THE NETHERLANDS,18
19,SWEDEN,20
24,ISRAEL,25
25,KOREA,26


In order to compare UVA to other colleges, we need to create a donation profile for each institution by summarizing the details provided in this data. I think we should be interested in total foreign gift amount, total donations, total countries, and donation composition (what types of donations compose the breakdown). Obviously, there are average (or proportion) values related to most of these counts, and we can easily produce these once we count the stated metrics. At this point, there is not an easy and reliable way to aggregate giftor and country names, so we won't be aggregating these yet; I may circle back to it later, though, when we narrow down what colleges to compare UVA to. Finally, I do not see an obvious benefit to including gift date in our analysis. I am assuming all of these gifts occur in a specified (and limited) period, which means the only insights we could gain are more seasonal (i.e. does UVA receive less winter donations than similar institutions?). The eye test tells me that seasonal patterns are not so important, but I may still consider it later.

# Global Analysis

## Average Country Rank of Giftor (Countries Weighted by Donation Amount)

First, lets first understand the landscape of donors by country to see if UVA is collecting "rare" money:

In [20]:
median_rank = grouped_gift["average_giftor_rank"].median()
mean_rank = grouped_gift["average_giftor_rank"].mean()
uva_rank = uva["average_giftor_rank"].iloc[0]

In [21]:
grouped_gift["average_giftor_rank"].describe()

count    319.000000
mean      14.799487
std       13.158876
min        1.000000
25%        6.042857
50%       12.197081
75%       18.385233
max      112.000000
Name: average_giftor_rank, dtype: float64

In [22]:
fig13 = px.histogram(grouped_gift["average_giftor_rank"], title = "Average Country Rank of Foreign Giftors", subtitle = "318 Schools",
                    labels={"value": "Average Donation Rank"}).add_vline(
                        mean_rank, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            median_rank, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                               uva_rank, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange")
                            )
fig13.update_layout(showlegend=False)
fig13.show()

In [23]:
uva["average_giftor_rank"]

256    39.04603
Name: average_giftor_rank, dtype: float64

In [24]:
np.quantile(grouped_gift["average_giftor_rank"], .95)

np.float64(39.940039268976115)

This is a somewhat startling finding. Namely, UVA is in the 95 percentile of average donation rank. That is, most of its money is coming from relatively rare sources. In some sense, they are not taking advantage of low hanging fruit--countries who donate a lot. 

We can see where UVA sits in the global landscape in several important metrics:

## Total Donation Amount

In [25]:
mean_gift_amt = grouped_gift["total_donation_amount"].mean()
median_gift_amt = grouped_gift["total_donation_amount"].median()
uva_gift_amt = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["total_donation_amount"].iloc[0]

In [26]:
mean_gift_amt

np.float64(52039233.5862069)

In [27]:
uva_gift_amt

np.int64(22189238)

In [28]:
np.quantile(grouped_gift["total_donation_amount"], 0.69)

np.float64(23097902.639999997)

In [29]:
grouped_gift["total_donation_amount"].describe()

count    3.190000e+02
mean     5.203923e+07
std      1.533565e+08
min      5.000000e+02
25%      1.573032e+06
50%      6.421819e+06
75%      3.151247e+07
max      1.477923e+09
Name: total_donation_amount, dtype: float64

In [30]:
fig1 = px.histogram(grouped_gift["total_donation_amount"], title = "Foreign Donations to US Colleges", subtitle = "318 Schools",
                    labels={"value": "Total Donation Amount ($)"}).add_vline(
                        mean_gift_amt, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            median_gift_amt, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                               uva_gift_amt, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange")
                            )
fig1.update_layout(showlegend=False)
fig1.show()

In [31]:
uva_gift_amt

np.int64(22189238)

In [32]:
median_gift_amt

np.float64(6421819.0)

Based on the histogram, we see that UVA, in the grand scheme of things, gets around the median in donations but less than the mean; it is also less than the third quartile (about the 69th percentile). Numerically, it is actually quite a bit larger than the median (22.2 million compared to 6.4 million), but these differences appear minimal with large outliers. We can remove those outliers and consider the same plot; note that we can remove these outliers because we are concerned with comparing UVA to the general global trends, and these exceptional are not very representative. 

In [33]:
small_school_donations = grouped_gift[grouped_gift["total_donation_amount"] < 100000000]

In [34]:
small_mean_gift_amt = small_school_donations["total_donation_amount"].mean()
small_median_gift_amt = small_school_donations["total_donation_amount"].median()

In [35]:
fig2 = px.histogram(small_school_donations["total_donation_amount"], title = "Foreign Donations to US Colleges", subtitle = "Schools with Less than 100 Million in Gifts",
                    labels={"value": "Total Donation Amount ($)"}).add_vline(
                        small_mean_gift_amt, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            small_median_gift_amt, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                               uva_gift_amt, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange")
                            )
fig2.update_layout(showlegend=False)
fig2.show()

## Total Donations

In [36]:
mean_gift_tot = grouped_gift["total_donations"].mean()
median_gift_tot = grouped_gift["total_donations"].median()
uva_gift_tot = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["total_donations"].iloc[0]

In [37]:
grouped_gift["total_donations"].mean()

np.float64(88.46708463949844)

In [38]:
grouped_gift["total_donations"].describe()

count     319.000000
mean       88.467085
std       295.377412
min         1.000000
25%         3.000000
50%        12.000000
75%        57.000000
max      3916.000000
Name: total_donations, dtype: float64

In [39]:
uva_gift_tot

np.int64(98)

In [40]:
np.quantile(grouped_gift["total_donations"], 0.83)

np.float64(100.88)

In [41]:
fig3 = px.histogram(grouped_gift["total_donations"], title = "Total Foreign Donations to US Colleges", subtitle = "318 Schools",
                    labels={"value": "Total Donations"}).add_vline(
                        mean_gift_tot, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            median_gift_tot, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                               uva_gift_tot, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange")
                            )
fig3.update_layout(showlegend=False)
fig3.show()

In [42]:
median_gift_tot

np.float64(12.0)

In [43]:
uva_gift_tot

np.int64(98)

Here, UVA sits comfortably above the third quartile (about 83rd percentile) and slightly above the mean; UVA receives a large volume of gifts, on average. Again, we can eliminate some outliers and reconsider the same plot; we can remove these outliers because we are concerned with comparing UVA to the general global trends, and these exceptional are not very representative. These discrepancies even further widen. 

In [44]:
small_donation_tots = grouped_gift[grouped_gift["total_donations"] < 500]

In [45]:
small_mean_gift_tot = small_donation_tots["total_donations"].mean()
small_median_gift_tot = small_donation_tots["total_donations"].median()

In [46]:
fig4 = px.histogram(small_donation_tots["total_donations"], title = "Total Foreign Donations to US Colleges", subtitle = "Schools with Less than 500 Total Donations",
                    labels={"value": "Total Donations"}).add_vline(
                        small_mean_gift_tot, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            small_median_gift_tot, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                               uva_gift_tot, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange")
                            )
fig4.update_layout(showlegend=False)
fig4.show()

From just these two sets of visuals, I am already beginning to get an idea about UVA. Namely, while they receive a relatively large number of donations, these donations are generally small, resulting in a more mediocre return than what the total donations would suggest. We can also see this using a scatter plot:

In [47]:
uva = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]

In [48]:
scatter_plt1 = px.scatter(
    grouped_gift,
    x="total_donation_amount",
    y="total_donations",
    hover_name="Institution Name",
    title=f"Total Foreign Donation Amount vs. Number of Donations Across 318 Schools",
    labels={
        "total_donation_amount": "Total Foreign Donation Amount ($)",
        "total_donations": "Number of Foreign Donations"
    }
)
median_x = grouped_gift["total_donation_amount"].median()
median_y = grouped_gift["total_donations"].median()

scatter_plt1.add_vline(
    x=median_x, line_dash="dash", line_color="gray",
    annotation_text=f"Median Amount: ${median_x:,.0f}", annotation_position="top right"
)
scatter_plt1.add_hline(
    y=median_y, line_dash="dash", line_color="gray",
    annotation_text=f"Median Count: {median_y:,.0f}", annotation_position="top right"
)

scatter_plt1.show()

In [49]:
df_for_plot = grouped_gift[(grouped_gift["total_donations"] < 500) & (grouped_gift["total_donation_amount"] < 100000000)]

In [50]:
scatter_plt2 = px.scatter(
    df_for_plot,
    x="total_donation_amount",
    y="total_donations",
    hover_name="Institution Name",
    title=f"Total Foreign Donation Amount vs. Number of Donations Across 318 Schools",
    labels={
        "total_donation_amount": "Total Foreign Donation Amount ($)",
        "total_donations": "Number of Foreign Donations"
    }
)
median_x = df_for_plot["total_donation_amount"].median()
median_y = df_for_plot["total_donations"].median()

scatter_plt2.add_vline(
    x=median_x, line_dash="dash", line_color="gray",
    annotation_text=f"Median Amount: ${median_x:,.0f}", annotation_position="top right"
)
scatter_plt2.add_hline(
    y=median_y, line_dash="dash", line_color="gray",
    annotation_text=f"Median Count: {median_y:,.0f}", annotation_position="top right"
)

scatter_plt2.add_trace(go.Scatter(x=uva["total_donation_amount"], y=uva["total_donations"], mode="markers", name="UVA", marker=dict(size=12, color="red", symbol="star")))

scatter_plt2.show()

There are clearly a lot more values to the right of UVA than above it.

# Average Donation Amount

As a natural extension of our discussion from the first two sets of visuals, we can consider the average donation amount UVA receives globally. Before comparing to other colleges, it may helpful to first look at the distribution of UVA's donation amounts:

In [51]:
uva

,OPEID,Institution Name,total_donation_amount,total_donations,total_countries,total_contract_donations,total_gift_donations,total_estate_donation,average_giftor_rank,total_institutions
256,374500,University of Virginia,22189238,98,15,64,34,0,39.04603,58


In [52]:
uva_raw = giftdf[giftdf["Institution Name"] == "University of Virginia"]

In [53]:
uva_mean_gift_amts = uva_raw["Foreign Gift Amount"].mean()
uva_median_gift_amts = uva_raw["Foreign Gift Amount"].median()

In [54]:
uva_raw["Foreign Gift Amount"].describe()

count    9.800000e+01
mean     2.264208e+05
std      3.433051e+05
min      3.100000e+01
25%      9.450000e+03
50%      8.053550e+04
75%      2.768882e+05
max      1.708021e+06
Name: Foreign Gift Amount, dtype: float64

In [55]:
fig5 = px.histogram(uva_raw["Foreign Gift Amount"], title = "Foreign Donations to UVA", subtitle="98 Donations",
                    labels={"value": "Donation Amount ($)"}).add_vline(
                         uva_mean_gift_amts, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            uva_median_gift_amts, annotation_text="Median", annotation_position ="top left", line_color = "black")

fig5.update_layout(showlegend=False)
fig5.show()

UVA's donation breakdown is skewed right, with the majority of its donating being less than 200,000. This confirms our original belief that UVA is generally a high volume donation school but most of these donations are not sizeable relatively speaking. We can confirm this by inspecting UVA's position globally in average donation amount:

In [56]:
grouped_gift["average_donation_amount"] = grouped_gift["total_donation_amount"] / grouped_gift["total_donations"]

In [57]:
average_donation_amount_mean = grouped_gift["average_donation_amount"].mean()
average_donation_amount_median = grouped_gift["average_donation_amount"].median()

In [58]:
uva_mean_gift_amts

np.float64(226420.79591836734)

In [59]:
np.quantile(grouped_gift["average_donation_amount"], 0.17)

np.float64(226500.01627997617)

In [60]:
grouped_gift["average_donation_amount"].describe()

count    3.190000e+02
mean     1.079061e+06
std      2.004926e+06
min      5.000000e+02
25%      2.997500e+05
50%      5.000000e+05
75%      1.091441e+06
max      2.270552e+07
Name: average_donation_amount, dtype: float64

In [61]:
fig6 = px.histogram(grouped_gift["average_donation_amount"], title = "Average Foreign Donations", subtitle="318 Schools",
                    labels={"value": "Average Donation Amount ($)"}).add_vline(
                         average_donation_amount_mean, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            average_donation_amount_median, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_mean_gift_amts, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig6.update_layout(showlegend=False)
fig6.show()

UVA is actually below the first quartile (about the 18th percentile). Even if we control for those exceptional outliers, UVA struggles in this department.

In [62]:
small_averages = grouped_gift[grouped_gift["average_donation_amount"] < 5000000]

In [63]:
small_averages_mean = small_averages["average_donation_amount"].mean()
small_averages_median = small_averages["average_donation_amount"].median()

In [64]:
fig7 = px.histogram(small_averages["average_donation_amount"], title = "Average Foreign Donations", subtitle="Schools with Average Donations less than 5 Million",
                    labels={"value": "Average Donation Amount ($)"}).add_vline(
                         small_averages_mean, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            small_averages_median, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_mean_gift_amts, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig7.update_layout(showlegend=False)
fig7.show()

# Giftors

Another possible consideration is the total number of giftors:

In [65]:
uva_giftors = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["total_countries"].iloc[0]
median_giftors = grouped_gift["total_countries"].median()
mean_giftors = grouped_gift["total_countries"].mean()

In [66]:
median_giftors

np.float64(3.0)

In [67]:
uva_giftors

np.int64(15)

In [68]:
np.quantile(grouped_gift["total_countries"], 0.82)

np.float64(15.0)

In [69]:
grouped_gift["total_countries"].describe()

count    319.000000
mean       8.420063
std       12.002582
min        1.000000
25%        1.000000
50%        3.000000
75%       11.000000
max      115.000000
Name: total_countries, dtype: float64

In [70]:
fig8 = px.histogram(grouped_gift["total_countries"], title = "Total Gift Countries for US Colleges", subtitle="318 Schools",
                    labels={"value": "Total Gift Countries"}).add_vline(
                         mean_giftors, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            median_giftors, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_giftors, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig8.update_layout(showlegend=False)
fig8.show()

With 15 countries, UVA sits comfortably above the mean of 8.4, median of 3, and third quartile of 11 (about 82nd percentile). While strongly above average, it is still a part of the main cluster of data, just on the tail. Isolating the main cluster improves its position further:

In [71]:
small_giftors = grouped_gift[grouped_gift["total_countries"] < 30]["total_countries"]
small_giftors_mean = small_giftors.mean()
small_giftors_median = small_giftors.median()

In [72]:
fig9 = px.histogram(small_giftors, title = "Total Gift Countries for US Colleges", subtitle="Schools with Less than 30 Donations",
                    labels={"value": "Total Gift Countries"}).add_vline(
                         small_giftors_mean, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            small_giftors_median, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_giftors, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig9.update_layout(showlegend=False)
fig9.show()

It may also be worth looking at the average number of gifts per country:

In [73]:
grouped_gift["Gifts per Country"] = grouped_gift["total_donations"] / grouped_gift["total_countries"]

In [74]:
grouped_gift["Gifts per Country"].describe()

count    319.000000
mean       6.223126
std       10.467143
min        1.000000
25%        1.500000
50%        3.000000
75%        6.844444
max      105.837838
Name: Gifts per Country, dtype: float64

In [75]:
uva_gpc = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["Gifts per Country"].iloc[0]
mean_gpc = grouped_gift["Gifts per Country"].mean()
median_gpc =  grouped_gift["Gifts per Country"].median()

In [76]:
uva_gpc

np.float64(6.533333333333333)

In [77]:
np.quantile(grouped_gift["Gifts per Country"], 0.74)

np.float64(6.397333333333331)

In [78]:
fig12 = px.histogram(grouped_gift["Gifts per Country"], title = "Total Gifts per Country for US Colleges", subtitle="318 Schools",
                    labels={"value": "Total Gifts per Country"}).add_vline(
                         mean_gpc, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            median_gpc, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_gpc, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig12.update_layout(showlegend=False)
fig12.show()

Again, UVA (6.53) sits comfortably above the median (3) and just below the third quartile (6.84), suggesting they do receive quite a few gifts from each country they interact with.

## Institutions per Country

In [79]:
uva["total_institutions"] / uva["total_countries"]

256    3.866667
dtype: float64

In [80]:
(grouped_gift["total_institutions"] / grouped_gift["total_countries"]).describe()

count    319.000000
mean       2.054450
std        4.300959
min        0.000000
25%        1.000000
50%        1.333333
75%        2.103679
max       71.297297
dtype: float64

In [81]:
np.quantile((grouped_gift["total_institutions"] / grouped_gift["total_countries"]), 0.93)

np.float64(3.865878787878788)

## Gift Type

Finally, we can look into a gift type comparison. Recall that there are three gift types (Monetary Gift, Contract, and Real Estate), and one of the gift types, Real Estate, occurs only 11 times (mostly between Japan and Temple University...Strange)

In [82]:
giftdf["Gift Type"].value_counts()

Gift Type
Contract         17274
Monetary Gift    10936
Real Estate         11
Name: count, dtype: int64

In [83]:
giftdf[giftdf["Gift Type"] == "Real Estate"]

,ID,OPEID,Institution Name,City,State,Foreign Gift Received Date,Foreign Gift Amount,Gift Type,Country of Giftor,Giftor Name,average rank
9245,9246,132000,"University of California, Santa Barbara",Santa Barbara,CA,2019-01-01,39456,Real Estate,JAPAN,Mitsubishi Chemical Holdings Inc,315648
9618,9619,135000,Colorado State University,Fort Collins,CO,2014-09-29,4312000,Real Estate,MEXICO,MIRA,133672000
23417,23418,337100,Temple University,Philadelphia,PA,2017-03-31,878139,Real Estate,JAPAN,Itochu Urban Community,7025112
23418,23419,337100,Temple University,Philadelphia,PA,2017-08-01,667555,Real Estate,JAPAN,Sublease,5340440
23419,23420,337100,Temple University,Philadelphia,PA,2017-08-01,331378,Real Estate,JAPAN,Sublease,2651024
23420,23421,337100,Temple University,Philadelphia,PA,2017-08-04,2829286,Real Estate,JAPAN,Taisei Biru Kanri K.K.,22634288
23421,23422,337100,Temple University,Philadelphia,PA,2017-08-04,1414186,Real Estate,JAPAN,Taisei Biru Kanri K.K.,11313488
23441,23442,337100,Temple University,Philadelphia,PA,2018-08-23,322314,Real Estate,JAPAN,Uninest,2578512
23452,23453,337100,Temple University,Philadelphia,PA,2019-03-31,409771,Real Estate,JAPAN,Itochu Urban Community,3278168
23457,23458,337100,Temple University,Philadelphia,PA,2019-06-30,3023538,Real Estate,ITALY,Fattura S.E.RO. CE. S.R.L.,81635526


I will construct visuals for the remaining two proportions but note that they are more or less complements of one another.

In [84]:
grouped_gift["prop_contract"] = grouped_gift["total_contract_donations"] / grouped_gift["total_donations"]
grouped_gift["prop_monetary"] = grouped_gift["total_gift_donations"] / grouped_gift["total_donations"]

In [85]:
contract_mean = grouped_gift["prop_contract"].mean()
contract_median = grouped_gift["prop_contract"].median()
uva_prop = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["prop_contract"].iloc[0]

In [86]:
uva_prop

np.float64(0.6530612244897959)

In [87]:
grouped_gift["prop_contract"].describe()

count    319.000000
mean       0.707758
std        0.375700
min        0.000000
25%        0.500000
50%        0.914894
75%        1.000000
max        1.000000
Name: prop_contract, dtype: float64

In [88]:
fig10 = px.histogram(grouped_gift["prop_contract"], title = "Proportion of Foreign Gifts that are Contracts", subtitle="318 Schools",
                    labels={"value": "Contract Proportion"}).add_vline(
                         contract_mean, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            contract_median, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_prop, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig10.update_layout(showlegend=False)
fig10.show()

In [89]:
gift_mean = grouped_gift["prop_monetary"].mean()
gift_median = grouped_gift["prop_monetary"].median()
uva_prop_gift = grouped_gift[grouped_gift["Institution Name"] == "University of Virginia"]["prop_monetary"].iloc[0]

In [90]:
grouped_gift["prop_monetary"].describe()

count    319.000000
mean       0.291671
std        0.375981
min        0.000000
25%        0.000000
50%        0.079365
75%        0.500000
max        1.000000
Name: prop_monetary, dtype: float64

In [91]:
np.quantile(grouped_gift["prop_monetary"], 0.68)

np.float64(0.36866995073891623)

In [92]:
fig11 = px.histogram(grouped_gift["prop_monetary"], title = "Proportion of Foreign Gifts that are Gifts", subtitle="318 Schools",
                    labels={"value": "Gift Proportion"}).add_vline(
                         gift_mean, annotation_text="Mean", annotation_position ="right", line_color = "red", annotation_font = dict(color = "red")).add_vline(
                            gift_median, annotation_text="Median", annotation_position ="top left", line_color = "black").add_vline(
                              uva_prop_gift, annotation_text="UVA", annotation_position ="bottom right", line_color = "orange", line_dash = "dash", annotation_font = dict(color = "orange"))

fig11.update_layout(showlegend=False)
fig11.show()

These visuals again tell an important story. While the median contract proportion is 0.91, UVA only receives 0.65 of their funding as contracts (which is above the first quartile). Conversely, about 0.35 of their funding comes from monetary gifts, which far surpasses the 0.08 global median (but less than the third quartile, about 68th percentile).

# Global Summary

As compared to the rest of colleges, UVA receives above average funding, which is comprised mainly of high-volume, small-quantity monetary gifts from 15 countries; these countries are relatively frugal elsewhere, giving UVA a 95th percentile in its average donation rank. While having both higher than average gift counts and higher than average countries, UVA still has a relatively high number of gifts per country; this suggests atleast a well-frequented connection and established relationship between the school and the countries it interacts with. 

# Paired Comparisons

While the global analysis is good for contextualization, I think it is honestly too much data to dig deeper than histograms and averages. In particular, since our goal is to ultimately offer ways UVA might increase its funding, it is really important to control for the university's defining factors before we dig deeper. For instance, globally renowned, private institutions like Harvard or MIT have a different set of tools at their disposal than UVA. Thus, controlling for a set of factors and comparing more deeply within this cohort can show us what is possible for the university.  To do so, I will be pulling in data from the U.S. Department of Education's College Scorecard. This dataset contains a lot of information, but I think the most pertinent for our evaluation comes from size (as represented by grad and undergrad population), region (as represented by Census designated region), its Carnegie Classifcation (roughly distinguishes between research/PhD heavy, masters dominated, and undergrad dominated institutions), its public/private status, and its status/rigor (as proxied by its standardized testing entry). Note that donations took place in the time period 2014 to 2020, so I will use college data from the 2017 to 2018 period, which is roughly in the middle. 

In [93]:
college_info = pd.read_csv("MERGED2017_18_PP.csv")
college_info = college_info[["OPEID", "UGDS", "GRADS", "REGION", "CCBASIC", "CONTROL", "SAT_AVG", "ACTCMMID"]]
college_info["OPEID"] = college_info["OPEID"].astype("Int64")
df = pd.merge(grouped_gift, college_info, how="left")

/var/folders/wn/4vhmdq557_7f8325xdqxk2tr0000gn/T/ipykernel_8455/2405995849.py:1: DtypeWarning: Columns (1729,1909,1910,1911,1912,1913) have mixed types. Specify dtype option on import or set low_memory=False.
  college_info = pd.read_csv("MERGED2017_18_PP.csv")


In [94]:
uva = df[df["Institution Name"] == "University of Virginia"]

A standard benchmark, in terms of similarity, is within one standard deviation. However, take a look at the SAT Average histogram.

In [95]:
px.histogram(df["SAT_AVG"])

To me, I see a bimodal distribution, where most colleges in the dataset are a part of a slightly skewed right distribution centered around 1100 or 1200. Then, there appears to be a second peak slightly above 1400, which is shaped more normally. The tails of these distributions overlap, and, UVA, with an average SAT of 1400, sits on the left side of this smaller group. With this in mind, I want to be cautious with the range I set for the SAT for several reasons. I think a naked standard deviation is going overinflate the spread because we principally have two distributions in one. As a result, I will artificially cut values at 1250 and recompute the standard deviation of this region. This is obviously not perfect. 

This bimodal issue doesn't really appear elsewhere

In [96]:
px.histogram(df["UGDS"])

In [97]:
ugds_std = df["UGDS"].std()
uva_ugds = uva["UGDS"].iloc[0]

In [98]:
sat_std = df[df["SAT_AVG"] >= 1250]["SAT_AVG"].std()
uva_sat = uva["SAT_AVG"].iloc[0]

In [99]:
ugds_mask = (((uva_ugds - 1 * ugds_std) <= df["UGDS"]) & (df["UGDS"] <= (uva_ugds + 1 * ugds_std)))

In [100]:
sat_mask = (((uva_sat - 1 * sat_std) <= df["SAT_AVG"]) & (df["SAT_AVG"] <= (uva_sat + 1 * sat_std)))

In [101]:
mask = ((df["REGION"].isin([1,2,5])) & (df["CONTROL"] == 1) & ugds_mask & sat_mask)

In [102]:
df[mask]

,OPEID,Institution Name,total_donation_amount,total_donations,total_countries,total_contract_donations,total_gift_donations,total_estate_donation,average_giftor_rank,total_institutions,...,Gifts per Country,prop_contract,prop_monetary,UGDS,GRADS,REGION,CCBASIC,CONTROL,SAT_AVG,ACTCMMID
68,156900,Georgia Institute of Technology,77979361,203,22,173,30,0,12.207197,85,...,9.227273,0.852217,0.147783,14810.0,13803.0,5.0,NaN,1.0,1389.0,32.0
189,297200,North Carolina State University,23762788,14,10,14,0,0,16.850156,12,...,1.400000,1.000000,0.000000,22801.0,10282.0,5.0,NaN,1.0,1331.0,29.0
190,297400,University of North Carolina - Chapel Hill,41292544,286,20,267,19,0,19.017717,49,...,14.300000,0.933566,0.066434,18658.0,11049.0,5.0,NaN,1.0,1393.0,31.0
229,337900,University of Pittsburgh,49024123,290,25,233,57,0,18.547830,86,...,11.600000,0.803448,0.196552,19134.0,9316.0,2.0,NaN,1.0,1349.0,30.0
238,342500,Clemson University,4912287,94,5,79,15,0,15.450555,8,...,18.800000,0.840426,0.159574,19172.0,4985.0,5.0,NaN,1.0,1338.0,29.0
261,370500,College of William & Mary,3384498,17,3,0,17,0,3.902167,5,...,5.666667,0.000000,1.000000,6243.0,2455.0,5.0,NaN,1.0,1406.0,31.0
263,374500,University of Virginia,22189238,98,15,64,34,0,39.046030,58,...,6.533333,0.653061,0.346939,16207.0,7705.0,5.0,NaN,1.0,1415.0,31.0


This passes the eye test and gives us about six different schools to compare with UVA in greater detail. Let's do some bar charts for some of the previous features to see where UVA sits:

In [103]:
selected_schools = df[mask]

In [104]:
smallfig1 = px.bar(selected_schools, x="Institution Name", y="total_donations")
smallfig1.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig1.show()

In [105]:
selected_schools["total_donations"].describe()

count      7.000000
mean     143.142857
std      117.332048
min       14.000000
25%       55.500000
50%       98.000000
75%      244.500000
max      290.000000
Name: total_donations, dtype: float64

In [106]:
smallfig2 = px.bar(selected_schools, x="Institution Name", y="total_donation_amount")
smallfig2.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig2.show()

In [107]:
selected_schools["total_donation_amount"].describe()

count    7.000000e+00
mean     3.179212e+07
std      2.647850e+07
min      3.384498e+06
25%      1.355076e+07
50%      2.376279e+07
75%      4.515833e+07
max      7.797936e+07
Name: total_donation_amount, dtype: float64

In [108]:
smallfig3 = px.bar(selected_schools, x="Institution Name", y="prop_contract")
smallfig3.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig3.show()

In [109]:
selected_schools["prop_contract"].describe()

count    7.000000
mean     0.726103
std      0.338028
min      0.000000
25%      0.728255
50%      0.840426
75%      0.892892
max      1.000000
Name: prop_contract, dtype: float64

In [110]:
smallfig4 = px.bar(selected_schools, x="Institution Name", y="total_countries")
smallfig4.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig4.show()

In [111]:
selected_schools["total_countries"].describe()

count     7.000000
mean     14.285714
std       8.557926
min       3.000000
25%       7.500000
50%      15.000000
75%      21.000000
max      25.000000
Name: total_countries, dtype: float64

In [112]:
smallfig5 = px.bar(selected_schools, x="Institution Name", y="Gifts per Country")
smallfig5.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig5.show()

In [113]:
selected_schools["Gifts per Country"].describe()

count     7.000000
mean      9.646753
std       5.816201
min       1.400000
25%       6.100000
50%       9.227273
75%      12.950000
max      18.800000
Name: Gifts per Country, dtype: float64

In [114]:
smallfig6 = px.bar(selected_schools, x="Institution Name", y="average_donation_amount")
smallfig6.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig6.show()

In [115]:
smallfig7 = px.bar(
    selected_schools,
    x="Institution Name",
    y="average_giftor_rank"
)

smallfig7.update_layout(
    xaxis={'categoryorder': 'total ascending'},
    yaxis_title="Average Giftor Rank"
)
smallfig7.show()

In [116]:
selected_schools["institutions per country"] = selected_schools["total_institutions"] / selected_schools["total_countries"]

/var/folders/wn/4vhmdq557_7f8325xdqxk2tr0000gn/T/ipykernel_8455/3736234245.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_schools["institutions per country"] = selected_schools["total_institutions"] / selected_schools["total_countries"]


In [117]:
smallfig8 = px.bar(selected_schools, x="Institution Name", y="institutions per country")
smallfig8.update_layout(xaxis={'categoryorder': 'total ascending'})
smallfig8.show()

In [118]:
smallfig7 = px.bar(selected_schools, x="Institution Name", y="average_giftor_rank")
smallfig7.update_layout(xaxis={'categoryorder': 'total ascending'})

It is amazing how conditioning on school background changes our view. Globally speaking, UVA had moderately above average donation amount, strong gift count, strongly below average amount per donation, strong country count, and below average contract proportion. In this narrowed group, though, UVA's picture shifts dramatically. Its donation amount is below the median (by about one million) and mean (by about 8 million, though the outlier of Georgia Tech  makes this worse) and its gift count is the median (surrounded by extremes on both sides). Additionally, its contract proportion is still quite low (but not as bad), and its gifts per country count is now below average. Surprisingly, its amount per donation is actually above average now. 

In [119]:
giftdf[giftdf["Institution Name"].isin(selected_schools["Institution Name"])].groupby("Country of Giftor")["Foreign Gift Amount"].mean().sort_values(ascending = False)

Country of Giftor
INDIA              2.196132e+06
KAZAKHSTAN         1.274758e+06
SCOTLAND           1.049486e+06
TANZANIA           9.254823e+05
DENMARK            7.990437e+05
SAUDI ARABIA       7.214105e+05
SWITZERLAND        5.603323e+05
BERMUDA            5.500000e+05
SWEDEN             5.283732e+05
GUERNSEY           5.000000e+05
ETHIOPIA           5.000000e+05
MALAWI             4.883075e+05
PANAMA             4.330148e+05
BANGLADESH         4.228589e+05
AUSTRALIA          4.139570e+05
SINGAPORE          4.025000e+05
CEYLON             3.866490e+05
CHINA              3.681677e+05
KOREA              3.206516e+05
SPAIN              3.149824e+05
IRELAND            2.948580e+05
BRAZIL             2.911620e+05
RWANDA             2.841350e+05
SOUTH KOREA        2.795541e+05
LEBANON            2.750000e+05
QATAR              2.624565e+05
ISLE OF MAN        2.598420e+05
TAIWAN             2.182108e+05
HONG KONG          1.890485e+05
FINLAND            1.769121e+05
COLOMBIA           1.7

In [120]:
giftdf[giftdf["Institution Name"].isin(selected_schools["Institution Name"])].groupby("Gift Type")["Foreign Gift Amount"].mean()

Gift Type
Contract         224154.513253
Monetary Gift    212189.494186
Name: Foreign Gift Amount, dtype: float64

In [121]:
cohort_raw = giftdf[giftdf["Institution Name"].isin(selected_schools["Institution Name"])]

Lets get some alluvial plots for donation streams in this cohort:

In [122]:
import plotly.graph_objects as go

# giftor = 'Giftor Name'
# recipi = 'Institution Name'
# flor = 'Foreign Gift Amount'

giftor = 'Giftor Name'
recipi = 'Institution Name'
flow = 'Foreign Gift Amount'
N = 30

flows = (
    cohort_raw.groupby([giftor, 
                recipi])
      [flow]
      .sum()
      .nlargest(N)
      .reset_index()
)

labels = (
    flows[giftor].tolist()
    + flows[recipi].tolist()
)

labels = list(dict.fromkeys(labels))

fig = go.Figure(
    go.Sankey(
        node=dict(label=labels),
        link=dict(
            source=flows[giftor]
                        .map(labels.index),
            target=flows[recipi]
                        .map(labels.index),
            value=flows[flow]
        )
    )
)

fig.show()

In [123]:
import plotly.graph_objects as go

# giftor = 'Giftor Name'
# recipi = 'Institution Name'
# flor = 'Foreign Gift Amount'

giftor = 'Country of Giftor'
recipi = 'Institution Name'
flow = 'Foreign Gift Amount'
N = 30

flows = (
    cohort_raw.groupby([giftor, 
                recipi])
      [flow]
      .sum()
      .nlargest(N)
      .reset_index()
)

labels = (
    flows[giftor].tolist()
    + flows[recipi].tolist()
)

labels = list(dict.fromkeys(labels))

fig = go.Figure(
    go.Sankey(
        node=dict(label=labels),
        link=dict(
            source=flows[giftor]
                        .map(labels.index),
            target=flows[recipi]
                        .map(labels.index),
            value=flows[flow]
        )
    )
)

fig.show()

Let's compute the breakdown of donations by country in the group:

In [124]:
donations_countries = cohort_raw.groupby("Country of Giftor")["Foreign Gift Amount"].sum()

In [125]:
bigs = ["CHINA", "ENGLAND", "SAUDI ARABIA", "JAPAN", "CANADA", "SWITZERLAND", "INDIA", "QATAR", "HONG KONG", "BERMUDA"]

In [126]:
cohort_raw[cohort_raw["Country of Giftor"].isin(bigs)]["Foreign Gift Amount"].sum() / cohort_raw["Foreign Gift Amount"].sum() 

np.float64(0.48791423107322657)

In [127]:
cohort_raw[cohort_raw["Country of Giftor"].isin(bigs) & (cohort_raw["Institution Name"] == "University of Virginia")]["Foreign Gift Amount"].sum() / cohort_raw[cohort_raw["Institution Name"] == "University of Virginia"]["Foreign Gift Amount"].sum()

np.float64(0.23059088374283065)

In [128]:
cohort_raw[cohort_raw["Country of Giftor"].isin(bigs) & (cohort_raw["Gift Type"] == "Contract")]["Foreign Gift Amount"].sum() / cohort_raw[(cohort_raw["Gift Type"] == "Contract")]["Foreign Gift Amount"].sum()

np.float64(0.44040753278587746)